# PINA tutorial: Burgers equation with sparse data

This notebook compares two ways to learn the 1D viscous Burgers equation:

1. A **data-only neural network** trained only on sparse measurements.
2. A **physics-informed neural network (PINN)** built with **PINA** and trained with:
   - sparse measurements,
   - the Burgers PDE residual,
   - the initial condition,
   - the boundary conditions.

The setup follows the standard PINN motivation highlighted by MathWorks: when measurements are sparse or only cover part of the domain, the physics term acts as a strong inductive bias and improves generalization.


## Problem statement

We solve the viscous Burgers equation on $(x, t) \in [-1, 1] \times [0, 1]$

$$
u_t + u u_x = \nu u_{xx}, \qquad \nu = \frac{0.01}{\pi}
$$

with

$$
u(x, 0) = -\sin(\pi x), \qquad u(-1, t)=u(1, t)=0.
$$

We first generate a simple finite-difference reference solution. That solution is only used to create synthetic measurements and to benchmark the models afterward.


In [ ]:
# If you run this notebook in Colab, uncomment the next line.
# !pip install "pina-mathlab[tutorial]"

import math
import random

import matplotlib.pyplot as plt
import torch

from pina import Condition, LabelTensor, Trainer
from pina.domain import CartesianDomain
from pina.equation import Equation
from pina.model import FeedForward
from pina.operator import grad, laplacian
from pina.problem import SpatialProblem, TimeDependentProblem
from pina.problem.zoo import SupervisedProblem
from pina.solver import PINN, SupervisedSolver

torch.manual_seed(7)
random.seed(7)

device = "cuda" if torch.cuda.is_available() else "cpu"
trainer_accelerator = "gpu" if torch.cuda.is_available() else "cpu"
torch.set_default_dtype(torch.float32)

print(f"device: {device}")


In [ ]:
nu = 0.01 / math.pi


def solve_burgers_reference(
    nx: int = 256,
    nt: int = 1001,
    t_final: float = 1.0,
    nu: float = nu,
):
    x = torch.linspace(-1.0, 1.0, nx)
    t = torch.linspace(0.0, t_final, nt)
    dx = x[1] - x[0]
    dt = t[1] - t[0]

    u = torch.zeros(nt, nx)
    u[0] = -torch.sin(math.pi * x)

    for n in range(nt - 1):
        u_n = u[n]
        u_next = u_n.clone()

        flux = 0.5 * u_n**2
        convection = -(dt / (2.0 * dx)) * (flux[2:] - flux[:-2])
        diffusion = nu * dt / dx**2 * (u_n[2:] - 2.0 * u_n[1:-1] + u_n[:-2])

        # Lax-Friedrichs stabilizes the nonlinear transport term enough for this tutorial grid.
        u_next[1:-1] = 0.5 * (u_n[2:] + u_n[:-2]) + convection + diffusion
        u_next[0] = 0.0
        u_next[-1] = 0.0
        u[n + 1] = u_next

    return x, t, u


x_ref, t_ref, u_ref = solve_burgers_reference()
X_ref, T_ref = torch.meshgrid(x_ref, t_ref, indexing="xy")

print("reference grid:", tuple(u_ref.shape))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

im = axes[0].imshow(
    u_ref,
    extent=[x_ref.min(), x_ref.max(), t_ref.max(), t_ref.min()],
    aspect="auto",
    cmap="coolwarm",
)
axes[0].set_title("Reference solution")
axes[0].set_xlabel("x")
axes[0].set_ylabel("t")
plt.colorbar(im, ax=axes[0], shrink=0.9)

for t_target in [0.0, 0.25, 0.50, 0.75, 1.0]:
    idx = torch.argmin(torch.abs(t_ref - t_target)).item()
    axes[1].plot(x_ref, u_ref[idx], label=f"t={t_ref[idx]:.2f}")

axes[1].set_title("A few time slices")
axes[1].set_xlabel("x")
axes[1].set_ylabel("u(x,t)")
axes[1].legend()
plt.tight_layout()


## Sparse measurements

To make the comparison meaningful, the supervised model only sees:

- a small number of points,
- mild observation noise,
- only the **early-time** part of the solution ($t \le 0.35$).

The PINN will use the same sparse measurements, but it will also see the PDE and the boundary/initial conditions throughout the full domain.


In [ ]:
def sample_measurements(
    x_grid,
    t_grid,
    u_grid,
    n_samples: int = 350,
    t_limit: float = 0.35,
    noise_std: float = 0.01,
):
    t_mask = t_grid <= t_limit
    candidate_t = torch.where(t_mask)[0]

    t_idx = candidate_t[torch.randint(len(candidate_t), (n_samples,))]
    x_idx = torch.randint(len(x_grid), (n_samples,))

    x = x_grid[x_idx]
    t = t_grid[t_idx]
    u = u_grid[t_idx, x_idx]

    noisy_u = u + noise_std * torch.randn_like(u)

    inputs = torch.stack([x, t], dim=1)
    targets = noisy_u.unsqueeze(1)
    return inputs, targets


train_inputs, train_targets = sample_measurements(x_ref, t_ref, u_ref)
train_inputs_lt = LabelTensor(train_inputs.clone(), ["x", "t"])
train_targets_lt = LabelTensor(train_targets.clone(), ["u"])

print("number of sparse measurements:", len(train_inputs))


In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(
    train_inputs[:, 0],
    train_inputs[:, 1],
    c=train_targets[:, 0],
    s=18,
    cmap="coolwarm",
    edgecolors="k",
    linewidths=0.2,
)
plt.xlabel("x")
plt.ylabel("t")
plt.title("Sparse and early-time measurements")
plt.colorbar(label="u")
plt.tight_layout()


## Baseline: data-only neural network in PINA

PINA also provides a supervised learning interface. We use that first so the comparison stays fair:

- same network family,
- same sparse observations,
- but **no** physics term.


In [ ]:
supervised_problem = SupervisedProblem(train_inputs_lt, train_targets_lt)

supervised_model = FeedForward(
    input_dimensions=2,
    output_dimensions=1,
    layers=[64, 64, 64],
    func=torch.nn.Tanh,
)

supervised_solver = SupervisedSolver(
    supervised_problem,
    supervised_model,
)

supervised_trainer = Trainer(
    supervised_solver,
    accelerator=trainer_accelerator,
    max_epochs=800,
    enable_checkpointing=False,
    enable_model_summary=False,
    logger=False,
)

supervised_trainer.train()


In [ ]:
eval_inputs = torch.stack([X_ref.reshape(-1), T_ref.reshape(-1)], dim=1)
eval_inputs_lt = LabelTensor(eval_inputs.clone(), ["x", "t"])

with torch.no_grad():
    u_supervised = supervised_solver(eval_inputs_lt.to(device)).cpu().reshape_as(u_ref)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

im0 = axes[0].imshow(
    u_supervised,
    extent=[x_ref.min(), x_ref.max(), t_ref.max(), t_ref.min()],
    aspect="auto",
    cmap="coolwarm",
)
axes[0].set_title("Data-only neural network")
axes[0].set_xlabel("x")
axes[0].set_ylabel("t")
plt.colorbar(im0, ax=axes[0], shrink=0.9)

im1 = axes[1].imshow(
    (u_supervised - u_ref).abs(),
    extent=[x_ref.min(), x_ref.max(), t_ref.max(), t_ref.min()],
    aspect="auto",
    cmap="magma",
)
axes[1].set_title("Absolute error")
axes[1].set_xlabel("x")
axes[1].set_ylabel("t")
plt.colorbar(im1, ax=axes[1], shrink=0.9)

plt.tight_layout()


## Proper PINN with PINA

Now we encode the Burgers equation directly in the loss:

- the **data** condition anchors the model to the sparse measurements,
- the **interior** condition enforces the PDE residual,
- the **initial** condition enforces $u(x,0)=-\sin(\pi x)$,
- the **boundary** conditions enforce $u(-1,t)=u(1,t)=0$.

This is the key difference from the previous model: the PINN gets supervision on the full spatio-temporal domain even where no labels are available.


In [ ]:
def burgers_equation(input_, output_):
    u = output_.extract(["u"])
    u_t = grad(output_, input_, components=["u"], d=["t"])
    u_x = grad(output_, input_, components=["u"], d=["x"])
    u_xx = laplacian(output_, input_, components=["u"], d=["x"])
    return u_t + u * u_x - nu * u_xx


def initial_condition(input_, output_):
    x = input_.extract(["x"])
    return output_.extract(["u"]) + torch.sin(math.pi * x)


def homogeneous_boundary(input_, output_):
    return output_.extract(["u"])


class BurgersProblem(TimeDependentProblem, SpatialProblem):
    output_variables = ["u"]
    spatial_domain = CartesianDomain({"x": [-1.0, 1.0]})
    temporal_domain = CartesianDomain({"t": [0.0, 1.0]})

    domains = {
        "interior": CartesianDomain({"x": [-1.0, 1.0], "t": [0.0, 1.0]}),
        "initial": CartesianDomain({"x": [-1.0, 1.0], "t": 0.0}),
        "left": CartesianDomain({"x": -1.0, "t": [0.0, 1.0]}),
        "right": CartesianDomain({"x": 1.0, "t": [0.0, 1.0]}),
    }

    conditions = {
        "data": Condition(input=train_inputs_lt, target=train_targets_lt),
        "interior": Condition(domain="interior", equation=Equation(burgers_equation)),
        "initial": Condition(domain="initial", equation=Equation(initial_condition)),
        "left": Condition(domain="left", equation=Equation(homogeneous_boundary)),
        "right": Condition(domain="right", equation=Equation(homogeneous_boundary)),
    }


pinn_problem = BurgersProblem()
pinn_problem.discretise_domain(n=4000, mode="random", domains=["interior"])
pinn_problem.discretise_domain(n=256, mode="grid", domains=["initial", "left", "right"])


In [ ]:
pinn_model = FeedForward(
    input_dimensions=2,
    output_dimensions=1,
    layers=[64, 64, 64],
    func=torch.nn.Tanh,
)

pinn_solver = PINN(pinn_problem, pinn_model)

pinn_trainer = Trainer(
    pinn_solver,
    accelerator=trainer_accelerator,
    max_epochs=1500,
    enable_checkpointing=False,
    enable_model_summary=False,
    logger=False,
)

pinn_trainer.train()


In [ ]:
with torch.no_grad():
    u_pinn = pinn_solver(eval_inputs_lt.to(device)).cpu().reshape_as(u_ref)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

im0 = axes[0].imshow(
    u_pinn,
    extent=[x_ref.min(), x_ref.max(), t_ref.max(), t_ref.min()],
    aspect="auto",
    cmap="coolwarm",
)
axes[0].set_title("PINN prediction")
axes[0].set_xlabel("x")
axes[0].set_ylabel("t")
plt.colorbar(im0, ax=axes[0], shrink=0.9)

im1 = axes[1].imshow(
    (u_pinn - u_ref).abs(),
    extent=[x_ref.min(), x_ref.max(), t_ref.max(), t_ref.min()],
    aspect="auto",
    cmap="magma",
)
axes[1].set_title("PINN absolute error")
axes[1].set_xlabel("x")
axes[1].set_ylabel("t")
plt.colorbar(im1, ax=axes[1], shrink=0.9)

plt.tight_layout()


In [ ]:
def relative_l2(u_true, u_pred):
    return torch.linalg.norm(u_true - u_pred) / torch.linalg.norm(u_true)


late_mask = t_ref >= 0.5
u_ref_late = u_ref[late_mask]
u_supervised_late = u_supervised[late_mask]
u_pinn_late = u_pinn[late_mask]

print("Relative L2 error on full domain")
print("  data-only :", float(relative_l2(u_ref, u_supervised)))
print("  PINN      :", float(relative_l2(u_ref, u_pinn)))
print()
print("Relative L2 error on late times (t >= 0.5)")
print("  data-only :", float(relative_l2(u_ref_late, u_supervised_late)))
print("  PINN      :", float(relative_l2(u_ref_late, u_pinn_late)))


In [ ]:
times_to_plot = [0.2, 0.6, 1.0]
fig, axes = plt.subplots(1, len(times_to_plot), figsize=(15, 4), sharey=True)

for ax, t_target in zip(axes, times_to_plot):
    idx = torch.argmin(torch.abs(t_ref - t_target)).item()
    ax.plot(x_ref, u_ref[idx], color="black", lw=2, label="reference")
    ax.plot(x_ref, u_supervised[idx], "--", lw=2, label="data-only")
    ax.plot(x_ref, u_pinn[idx], lw=2, label="PINN")
    ax.set_title(f"t = {t_ref[idx]:.2f}")
    ax.set_xlabel("x")
    ax.grid(alpha=0.3)

axes[0].set_ylabel("u(x,t)")
axes[0].legend()
plt.tight_layout()


## Takeaways

Typical behavior in this notebook:

- the **data-only** network fits the early sparse samples reasonably well,
- but it degrades when asked to predict outside the region covered by those samples,
- the **PINN** usually stays much closer to the true solution because the PDE, the initial condition, and the boundary conditions constrain the model everywhere.

That is the central value proposition of PINNs: they can turn limited measurements into a better global solution by injecting known physics into the training objective.


## Suggested exercises

Try changing one thing at a time:

1. Increase the measurement noise.
2. Reduce the number of sparse measurements.
3. Remove the PINN data condition and train from physics only.
4. Increase the viscosity and see how the learning problem changes.
5. Replace the feed-forward network depth and width and compare convergence.
